# 🧠 Word Embeddings — Complete Hands-On Tutorial
### Based on Lena Voita's NLP Course · lena-voita.github.io/nlp_course/word_embeddings.html

---

## 📚 What You Will Learn

| Section | Topic | Key Skill |
|---------|-------|----------|
| 1 | One-Hot Vectors | Building & problems |
| 2 | Co-occurrence Matrices | Counting context |
| 3 | PPMI | Pointwise Mutual Information |
| 4 | Word2Vec from Scratch | Neural prediction, gradients |
| 5 | Negative Sampling | Efficient training |
| 6 | Skip-Gram vs CBOW | Architecture differences |
| 7 | GloVe Loss | Understanding the objective |
| 8 | Semantic Analogies | king - man + woman = queen |
| 9 | t-SNE Visualization | Seeing semantic space |
| 10 | Cross-Lingual Embeddings | Linear mapping across languages |
| 11 | Bias in Embeddings | Detecting & debiasing |
| 12 | Semantic Change | Tracking meaning shifts |

---

> **💡 Analogy before we start:** Think of word embeddings like GPS coordinates for meaning. Just as latitude/longitude captures a city's position in geographic space, a word embedding captures a word's position in *semantic* space. Cities close together on a map share geography; words close in embedding space share meaning.

---

## ⚙️ Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from collections import Counter, defaultdict
from itertools import product
import warnings, math, random
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
random.seed(42)

# Try importing sklearn components we'll need
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# ── Colour palette ──────────────────────────────────────────────
BLUE   = '#1a3a5c'
ACCENT = '#4a90c4'
GREEN  = '#5a8a2c'
ORANGE = '#d97706'
RED    = '#c0392b'
PURPLE = '#7b2d8b'
GRAY   = '#555555'

def styled_print(title, content, color='blue'):
    colors = {'blue': '\033[94m', 'green': '\033[92m', 'red': '\033[91m',
              'yellow': '\033[93m', 'bold': '\033[1m', 'end': '\033[0m'}
    c = colors.get(color, '')
    print(f"\n{c}{'='*60}\033[0m")
    print(f"{colors['bold']}{title}\033[0m")
    print(f"{c}{'='*60}\033[0m")
    print(content)

print("✅ All imports successful! Let's build word embeddings from scratch.")

---
## 📖 Our Toy Corpus

Throughout this notebook we'll use a small, hand-crafted corpus covering **animals, food, and royalty** — three very different semantic domains. This lets us *see* whether our embeddings successfully separate these topics.

> **Analogy:** This corpus is like a tiny library. Even though it only has ~30 sentences, a smart reader (our model) should be able to deduce that "dog" and "cat" belong together, while "king" and "queen" form a different cluster.

In [ ]:
# ── Our toy corpus ──────────────────────────────────────────────
CORPUS = [
    "the dog barked at the cat",
    "the cat sat on the mat",
    "the dog and cat are pets",
    "dogs and cats are animals",
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king and queen are royals",
    "the prince will become king",
    "the princess will become queen",
    "man and woman are humans",
    "the man went to the market",
    "the woman went to the market",
    "dogs like to eat meat",
    "cats like to eat fish",
    "the king ate bread and meat",
    "bread and fish are food",
    "animals eat food to survive",
    "humans eat bread and meat and fish",
    "the dog ran in the park",
    "the cat ran in the garden",
    "the prince and princess are royals",
    "the king is a man",
    "the queen is a woman",
    "man has a dog as a pet",
    "woman has a cat as a pet",
    "the market sells bread and fish",
    "animals in the park include dogs and cats",
    "the kingdom has a king and a queen",
    "humans and animals both need food",
    "the prince is a young man",
    "the princess is a young woman",
]

# Tokenise
def tokenise(corpus):
    tokens = []
    for sent in corpus:
        tokens.extend(sent.lower().split())
    return tokens

tokens = tokenise(CORPUS)
freq   = Counter(tokens)

# Build vocabulary (words appearing >= 2 times, exclude very common stopwords for clarity)
STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'and', 'to', 'in', 'at', 'on', 'as', 'has',
             'both', 'will', 'of', 'have', 'be', 'has', 'went', 'ran', 'need', 'include'}
vocab_words = sorted([w for w, c in freq.items() if c >= 2 and w not in STOPWORDS])
word2idx    = {w: i for i, w in enumerate(vocab_words)}
idx2word    = {i: w for w, i in word2idx.items()}
V           = len(vocab_words)

print(f"Corpus statistics:")
print(f"  Sentences   : {len(CORPUS)}")
print(f"  Total tokens: {len(tokens)}")
print(f"  Vocabulary  : {V} words")
print(f"\nVocabulary:")
print(' | '.join(vocab_words))

# Group words by semantic category for later visualisation
CATEGORIES = {
    'animals' : ['dog', 'cat', 'dogs', 'cats'],
    'royalty' : ['king', 'queen', 'prince', 'princess', 'kingdom', 'royals'],
    'humans'  : ['man', 'woman', 'humans'],
    'food'    : ['bread', 'meat', 'fish', 'food', 'market'],
    'other'   : ['pets', 'animals', 'park', 'garden', 'young', 'survive'],
}
CAT_COLORS = {'animals': ACCENT, 'royalty': ORANGE, 'humans': GREEN, 'food': PURPLE, 'other': GRAY}

def get_color(word):
    for cat, words in CATEGORIES.items():
        if word in words:
            return CAT_COLORS[cat]
    return GRAY

---
## 1️⃣ One-Hot Vectors

The simplest representation: each word gets a vector of all zeros, except for a single `1` at its unique index position.

**The core problem:** Every pair of words has the same cosine similarity = **0**. The model literally cannot tell that `dog` is more similar to `cat` than to `bread`.

> **Analogy:** One-hot vectors are like assigning every person in the world a unique random phone number. The numbers carry no information about who those people are, what they look like, or how they are related.

In [ ]:
# ── Build one-hot vectors ────────────────────────────────────────
def one_hot(word, word2idx, V):
    """Return the one-hot vector for a word."""
    vec = np.zeros(V)
    if word in word2idx:
        vec[word2idx[word]] = 1.0
    return vec

# Demonstrate a few vectors
words_to_show = ['dog', 'cat', 'king', 'bread']
print("One-Hot Vectors (showing first 15 dimensions):")
print(f"{'Word':<12}  Vector[:15]")
print('-' * 55)
for w in words_to_show:
    if w in word2idx:
        v = one_hot(w, word2idx, V)
        print(f"{w:<12}  {v[:15].astype(int)}")

# Visualise the one-hot matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Matrix heatmap
sample = ['dog', 'cat', 'king', 'queen', 'man', 'bread']
matrix = np.array([one_hot(w, word2idx, V) for w in sample if w in word2idx])
valid  = [w for w in sample if w in word2idx]
ax = axes[0]
ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(len(valid)))
ax.set_yticklabels(valid, fontsize=11)
ax.set_xlabel('Dimension (Vocabulary Index)', fontsize=11)
ax.set_title('One-Hot Vectors\n(each row = one word)', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, V, max(1, V//6)))

# Pairwise cosine similarities
pairs = [(w1, w2) for w1 in valid for w2 in valid]
sim_matrix = np.array([[np.dot(one_hot(w1, word2idx, V), one_hot(w2, word2idx, V))
                        for w2 in valid] for w1 in valid])
ax2 = axes[1]
im = ax2.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(len(valid))); ax2.set_xticklabels(valid, rotation=45, fontsize=10)
ax2.set_yticks(range(len(valid))); ax2.set_yticklabels(valid, fontsize=10)
ax2.set_title('Pairwise Cosine Similarities\n(all zeros = no meaning captured!)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('/tmp/onehot.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n🔑 KEY INSIGHT: All cosine similarities = 0.")
print("   dog↔cat similarity == dog↔bread similarity (both 0!)")
print("   One-hot vectors capture ZERO semantic information.")

---
## 2️⃣ Co-Occurrence Matrix

The **distributional hypothesis**: *words that appear in similar contexts have similar meanings.*

We count: for every word `w`, how many times does each other word `c` appear within a window of ±`L` positions?

> **Analogy:** The co-occurrence matrix is like a social network adjacency matrix. If two people (words) keep showing up at the same events (contexts), they are likely connected in some meaningful way.

In [ ]:
# ── Build Co-Occurrence Matrix ────────────────────────────────────
def build_cooccurrence(corpus, word2idx, window=2):
    """
    Build a word x word co-occurrence matrix.
    M[i,j] = number of times word_i appears within `window` positions of word_j.
    """
    V = len(word2idx)
    M = np.zeros((V, V), dtype=np.float32)
    
    for sentence in corpus:
        words = sentence.lower().split()
        # Keep only words in vocabulary
        words = [w for w in words if w in word2idx]
        for i, center in enumerate(words):
            ci = word2idx[center]
            # Look left and right within window
            for j in range(max(0, i - window), min(len(words), i + window + 1)):
                if i != j:
                    cj = word2idx[words[j]]
                    M[ci, cj] += 1
    return M

M_raw = build_cooccurrence(CORPUS, word2idx, window=2)

# Visualise for a subset of interesting words
key_words = ['dog', 'cat', 'king', 'queen', 'man', 'woman', 'bread', 'fish', 'meat', 'prince']
key_idx   = [word2idx[w] for w in key_words if w in word2idx]
valid_kw  = [w for w in key_words if w in word2idx]
sub_M     = M_raw[np.ix_(key_idx, key_idx)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw matrix
im1 = axes[0].imshow(sub_M, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(valid_kw))); axes[0].set_xticklabels(valid_kw, rotation=45, fontsize=10)
axes[0].set_yticks(range(len(valid_kw))); axes[0].set_yticklabels(valid_kw, fontsize=10)
axes[0].set_title('Raw Co-Occurrence Counts (window=2)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[0])
# Annotate cells
for i in range(len(valid_kw)):
    for j in range(len(valid_kw)):
        axes[0].text(j, i, int(sub_M[i,j]), ha='center', va='center', fontsize=8,
                    color='black' if sub_M[i,j] < sub_M.max()*0.6 else 'white')

# Cosine similarity of raw count vectors
norm_M   = normalize(M_raw, norm='l2')
sub_norm = norm_M[np.ix_(key_idx, key_idx)]
sim_raw  = sub_norm @ sub_norm.T

im2 = axes[1].imshow(sim_raw, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks(range(len(valid_kw))); axes[1].set_xticklabels(valid_kw, rotation=45, fontsize=10)
axes[1].set_yticks(range(len(valid_kw))); axes[1].set_yticklabels(valid_kw, fontsize=10)
axes[1].set_title('Cosine Similarity from Co-Occurrence Vectors', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[1])
for i in range(len(valid_kw)):
    for j in range(len(valid_kw)):
        axes[1].text(j, i, f"{sim_raw[i,j]:.2f}", ha='center', va='center', fontsize=7,
                    color='black' if sim_raw[i,j] < 0.6 else 'white')

plt.tight_layout()
plt.savefig('/tmp/cooccurrence.png', dpi=120, bbox_inches='tight')
plt.show()

# Show most similar words
print("\n📊 Most Similar Words (by Co-Occurrence Cosine Similarity):")
for query in ['dog', 'king', 'bread']:
    if query in word2idx:
        qi   = word2idx[query]
        sims = norm_M[qi] @ norm_M.T
        top5 = sorted([(idx2word[i], sims[i]) for i in range(V) if i != qi], key=lambda x: -x[1])[:5]
        print(f"  '{query}' → " + ', '.join(f"{w}({s:.2f})" for w, s in top5))

---
## 3️⃣ Positive Pointwise Mutual Information (PPMI)

**Problem with raw counts:** Frequent words like "the", "is" co-occur with everything, creating spurious high counts that pollute our vectors.

**PMI solution:** Measure whether two words co-occur *more than you'd expect by chance*.

$$\text{PMI}(w, c) = \log \frac{P(w, c)}{P(w) \cdot P(c)}$$

**PPMI** = `max(PMI, 0)` — we only keep positive associations (negative PMI values are unreliable on small corpora).

> **Analogy:** PMI is like a "surprise score" for co-occurrence. If 10% of people are doctors and 10% like jazz, you'd expect 1% to be doctor-jazz fans by chance. If you observe 5%, that's a surprise factor of log(5) ≈ 1.6. PPMI only records pleasant surprises, not disappointing under-representations.

In [ ]:
# ── PPMI Matrix ──────────────────────────────────────────────────
def compute_ppmi(M):
    """
    Compute Positive Pointwise Mutual Information matrix from co-occurrence counts.
    PPMI(w,c) = max(0, log[ P(w,c) / (P(w) * P(c)) ])
    """
    total   = M.sum()                          # total co-occurrence events
    row_sum = M.sum(axis=1, keepdims=True)      # P(w) unnormalised
    col_sum = M.sum(axis=0, keepdims=True)      # P(c) unnormalised
    
    # Avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log2((M * total) / (row_sum * col_sum + 1e-9) + 1e-9)
    
    ppmi = np.maximum(pmi, 0)                  # only keep positive values
    return ppmi

M_ppmi = compute_ppmi(M_raw)

# Compare: raw count vs PPMI for the same subset
sub_ppmi = M_ppmi[np.ix_(key_idx, key_idx)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im1 = axes[0].imshow(sub_M, cmap='YlOrRd')
axes[0].set_title('Raw Co-Occurrence Counts', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(valid_kw))); axes[0].set_xticklabels(valid_kw, rotation=45, fontsize=9)
axes[0].set_yticks(range(len(valid_kw))); axes[0].set_yticklabels(valid_kw, fontsize=9)
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(sub_ppmi, cmap='Blues')
axes[1].set_title('PPMI Matrix (surprisal-weighted)', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(len(valid_kw))); axes[1].set_xticklabels(valid_kw, rotation=45, fontsize=9)
axes[1].set_yticks(range(len(valid_kw))); axes[1].set_yticklabels(valid_kw, fontsize=9)
for i in range(len(valid_kw)):
    for j in range(len(valid_kw)):
        v = sub_ppmi[i,j]
        axes[1].text(j, i, f"{v:.1f}", ha='center', va='center', fontsize=7,
                    color='white' if v > sub_ppmi.max()*0.6 else 'black')
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Raw Counts vs PPMI: Frequency Bias is Removed in PPMI', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/ppmi.png', dpi=120, bbox_inches='tight')
plt.show()

# Show nearest neighbours using PPMI
norm_ppmi = normalize(M_ppmi, norm='l2')
print("\n📊 Nearest Neighbours via PPMI Cosine Similarity:")
for query in ['dog', 'king', 'man', 'bread']:
    if query in word2idx:
        qi   = word2idx[query]
        sims = norm_ppmi[qi] @ norm_ppmi.T
        top5 = sorted([(idx2word[i], sims[i]) for i in range(V) if i != qi and sims[i] > 0],
                      key=lambda x: -x[1])[:5]
        print(f"  '{query:<7}' → " + ', '.join(f"{w}({s:.2f})" for w, s in top5))

print("\n✅ PPMI captures semantics: 'king' now neighbours 'queen', not just frequent filler words!")

---
## 3b️⃣ SVD Dimensionality Reduction (LSA-style)

The PPMI matrix is `V × V` — huge and sparse. **Truncated SVD** compresses it to `V × d` (d = 50 or 100), capturing the most important latent dimensions of variation.

This is the essence of **Latent Semantic Analysis (LSA)**.

$$M_{PPMI} \approx U_d \cdot \Sigma_d \cdot V_d^T$$

We use $U_d \cdot \sqrt{\Sigma_d}$ as our word vectors (eigenvalue weighting — a trick from Levy et al. 2015).

In [ ]:
# ── SVD / LSA on PPMI Matrix ─────────────────────────────────────
def svd_embeddings(M, n_components=10):
    """
    Reduce M via Truncated SVD.
    Returns word embeddings = U_d * sqrt(Sigma_d)  (eigenvalue weighting)
    """
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    U   = svd.fit_transform(M)          # U * Sigma
    sig = np.sqrt(svd.singular_values_) # sqrt(Sigma)
    # eigenvalue weighting: U * sqrt(Sigma)
    embeddings = U / svd.singular_values_ * sig  
    return normalize(embeddings, norm='l2'), svd.explained_variance_ratio_

# How many dimensions explain how much variance?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_comp   = min(V - 1, 15)
svd_full   = TruncatedSVD(n_components=max_comp, random_state=42)
svd_full.fit(M_ppmi)
cumvar     = np.cumsum(svd_full.explained_variance_ratio_)

axes[0].bar(range(1, max_comp+1), svd_full.explained_variance_ratio_ * 100, color=ACCENT, alpha=0.8)
axes[0].set_xlabel('Component', fontsize=11)
axes[0].set_ylabel('Variance Explained (%)', fontsize=11)
axes[0].set_title('Variance per SVD Component\n(each component = one latent "topic")', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, max_comp+1), cumvar * 100, 'o-', color=GREEN, linewidth=2, markersize=6)
axes[1].axhline(80, color=ORANGE, linestyle='--', alpha=0.7, label='80% threshold')
axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Variance (%)', fontsize=11)
axes[1].set_title('Cumulative Variance Explained\n(how many dims do we really need?)', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/svd_variance.png', dpi=120, bbox_inches='tight')
plt.show()

# Build final count-based embeddings (LSA)
n_dim      = min(10, V-1)
E_lsa, _   = svd_embeddings(M_ppmi, n_components=n_dim)

print(f"\nLSA Embeddings shape: {E_lsa.shape}")
print(f"(V={V} words × d={n_dim} dimensions)")

# Nearest neighbours using LSA
sim_lsa = E_lsa @ E_lsa.T
print("\n📊 Nearest Neighbours via LSA:")
for query in ['dog', 'king', 'man', 'fish']:
    if query in word2idx:
        qi   = word2idx[query]
        sims = sim_lsa[qi]
        top5 = sorted([(idx2word[i], sims[i]) for i in range(V) if i != qi], key=lambda x: -x[1])[:5]
        print(f"  '{query:<6}' → " + ', '.join(f"{w}({s:.2f})" for w, s in top5))

---
## 4️⃣ Word2Vec from Scratch

Word2Vec is a neural approach. It trains word vectors by making them **predict their context words**.

### Architecture
- Each word has **two** vectors: `v_w` (when it's the center word) and `u_w` (when it's a context word)
- The probability of context word `o` given center word `c` is:

$$P(o|c) = \frac{\exp(u_o^T v_c)}{\sum_{w \in V} \exp(u_w^T v_c)}$$

- The loss for one (center, context) pair:
$$J = -\log P(o|c) = -u_o^T v_c + \log \sum_w \exp(u_w^T v_c)$$

> **Analogy:** Training Word2Vec is like drilling fill-in-the-blank exercises. For every sentence, you cover one word and ask: *"Given the surrounding words, which word fits here?"* After millions of such exercises, your word vectors must have learned enough about meaning to answer correctly.

In [ ]:
# ── Word2Vec: Data Preparation ───────────────────────────────────
def generate_training_pairs(corpus, word2idx, window=2):
    """
    Generate (center_word_idx, context_word_idx) pairs for Word2Vec.
    For each word in the corpus, we pair it with every word in its ±window neighbourhood.
    """
    pairs = []
    for sentence in corpus:
        words     = sentence.lower().split()
        idx_seq   = [word2idx[w] for w in words if w in word2idx]
        for i, center_idx in enumerate(idx_seq):
            for j in range(max(0, i - window), min(len(idx_seq), i + window + 1)):
                if i != j:
                    pairs.append((center_idx, idx_seq[j]))
    return pairs

training_pairs = generate_training_pairs(CORPUS, word2idx, window=2)
print(f"Generated {len(training_pairs):,} (center, context) training pairs")
print("\nSample pairs (word_idx → word):")
for c, ctx in training_pairs[:8]:
    print(f"  center='{idx2word[c]:<8}' context='{idx2word[ctx]}'")

In [ ]:
# ── Word2Vec: Softmax Probability & Loss ─────────────────────────
def softmax(x):
    """Numerically stable softmax."""
    x = x - x.max()   # subtract max for numerical stability
    e = np.exp(x)
    return e / e.sum()

def compute_loss_and_grads(center_idx, context_idx, W_center, W_context):
    """
    Compute negative log-likelihood loss and gradients for ONE (center, context) pair.
    
    W_center  : (V, d) – center word embedding matrix  (v_w vectors)
    W_context : (V, d) – context word embedding matrix (u_w vectors)
    
    Returns: (loss, grad_center_row, grad_context_matrix)
    """
    V, d      = W_center.shape
    v_c       = W_center[center_idx]          # shape (d,)
    u_o       = W_context[context_idx]         # shape (d,) – true context word
    
    # Compute scores: u_w^T v_c  for all words w
    scores    = W_context @ v_c               # shape (V,)
    probs     = softmax(scores)               # shape (V,)
    
    # Loss: -log P(o|c)
    loss      = -np.log(probs[context_idx] + 1e-9)
    
    # Gradient w.r.t. v_c (center word vector)
    # dJ/dv_c = -u_o + sum_w p_w * u_w  =  -(u_o - E[u])
    expected_context = probs @ W_context      # shape (d,) – weighted average context vector
    grad_vc           = expected_context - u_o
    
    # Gradient w.r.t. context vectors U  (only non-zero for all rows)
    # dJ/du_w = (p_w - 1{w=o}) * v_c
    delta           = probs.copy()             # shape (V,)
    delta[context_idx] -= 1.0
    grad_U          = np.outer(delta, v_c)    # shape (V, d)
    
    return loss, grad_vc, grad_U

# Quick demonstration
D = 4  # tiny embedding dim for demo
W_c_demo = np.random.randn(V, D) * 0.01
W_ctx_demo = np.random.randn(V, D) * 0.01

c_idx  = word2idx['king']
ctx_idx = word2idx['queen']
loss, gc, gU = compute_loss_and_grads(c_idx, ctx_idx, W_c_demo, W_ctx_demo)
print(f"Demo loss for ('king' → 'queen'): {loss:.4f}")
print(f"Gradient shape for center word  : {gc.shape}")
print(f"Gradient shape for context words: {gU.shape}")

# Visualise the softmax probability distribution
scores = W_ctx_demo @ W_c_demo[c_idx]
probs  = softmax(scores)

fig, ax = plt.subplots(figsize=(14, 4))
colors_bar = [RED if i == ctx_idx else ACCENT for i in range(V)]
ax.bar(range(V), probs, color=colors_bar, alpha=0.8)
ax.axhline(1/V, color=ORANGE, linestyle='--', alpha=0.7, label=f'Uniform (1/V={1/V:.3f})')
ax.set_xticks(range(V))
ax.set_xticklabels(vocab_words, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('P(context | center="king")', fontsize=11)
ax.set_title('Softmax Output: Probability Distribution over Context Words\n(Red = true context word "queen"; after random init, all are roughly equal)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/softmax_demo.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word2Vec: Full Training Loop (Skip-Gram, Softmax) ─────────────
def train_w2v_skipgram(training_pairs, V, d=20, lr=0.05, epochs=300, verbose=True):
    """
    Train Word2Vec Skip-Gram with full softmax.
    
    Parameters
    ----------
    training_pairs : list of (center_idx, context_idx)
    V              : vocabulary size
    d              : embedding dimensionality
    lr             : learning rate
    epochs         : number of full passes over training data
    """
    # Initialise with small random values (important! large init → bad gradients)
    W_center  = np.random.randn(V, d) * 0.01
    W_context = np.random.randn(V, d) * 0.01
    
    loss_history = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        # Shuffle pairs each epoch (SGD)
        shuffled = training_pairs.copy()
        random.shuffle(shuffled)
        
        for center_idx, context_idx in shuffled:
            loss, grad_vc, grad_U = compute_loss_and_grads(
                center_idx, context_idx, W_center, W_context
            )
            total_loss += loss
            
            # Gradient descent update
            W_center[center_idx]  -= lr * grad_vc
            W_context             -= lr * grad_U
        
        avg_loss = total_loss / len(shuffled)
        loss_history.append(avg_loss)
        
        if verbose and (epoch % 50 == 0 or epoch == epochs - 1):
            print(f"  Epoch {epoch+1:4d}/{epochs}  Loss: {avg_loss:.4f}")
    
    return W_center, W_context, loss_history

print("Training Word2Vec Skip-Gram (full softmax)...")
print(f"  Vocab size: {V}, Embedding dim: 20, Epochs: 300")
W_center, W_context, loss_hist = train_w2v_skipgram(
    training_pairs, V, d=20, lr=0.05, epochs=300
)

# After training, use only W_center as word embeddings
E_w2v = normalize(W_center, norm='l2')

# Plot training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_hist, color=ACCENT, linewidth=2)
ax.fill_between(range(len(loss_hist)), loss_hist, alpha=0.2, color=ACCENT)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Average Loss', fontsize=11)
ax.set_title('Word2Vec Training Curve\n(Loss decreasing = vectors learning context patterns)', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/w2v_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nFinal loss: {loss_hist[-1]:.4f}")

---
## 5️⃣ Negative Sampling

**Problem:** Full softmax requires computing scores for ALL V words at every step — O(V) per update.

**Solution:** Instead of normalizing over the entire vocabulary, sample K random "negative" words and only update those.

$$J_{neg}(\theta) = -\log\sigma(u_{ctx}^T v_c) - \sum_{k=1}^{K} \log\sigma(-u_{neg_k}^T v_c)$$

This is O(K) per update, where K = 2–20, vs O(V) = O(50,000).

> **Analogy:** Full softmax is like a student who, when asked "Is Paris the capital of France?", consults every country on earth before answering. Negative sampling is like consulting just a few randomly-chosen wrong answers: France→Germany? No. France→Japan? No. Therefore, France→Paris is probably right.

In [ ]:
# ── Negative Sampling ────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def build_neg_sampler(word2idx, corpus, power=0.75):
    """
    Build a negative sampling distribution: P(w) ∝ freq(w)^0.75
    The 0.75 power smooths the distribution, boosting rare words.
    """
    # Count word frequencies in corpus
    all_words = []
    for sent in corpus:
        all_words.extend([w for w in sent.lower().split() if w in word2idx])
    freq  = Counter(all_words)
    vocab = list(word2idx.keys())
    
    # P(w) ∝ freq^power
    counts = np.array([freq.get(w, 0) ** power for w in vocab])
    probs  = counts / counts.sum()
    
    return vocab, probs

def sample_negatives(center_idx, context_idx, vocab, probs, K=5):
    """Sample K negative (non-context) words."""
    negs = []
    while len(negs) < K:
        idx = np.random.choice(len(vocab), p=probs)
        if idx != center_idx and idx != context_idx:
            negs.append(idx)
    return negs

def neg_sampling_loss_and_grads(center_idx, context_idx, neg_indices, W_center, W_context):
    """
    Compute negative sampling loss and gradients for ONE (center, context, negatives) tuple.
    """
    v_c   = W_center[center_idx]                   # shape (d,)
    u_pos = W_context[context_idx]                  # shape (d,) – positive context
    
    # Positive term: log σ(u_pos · v_c)
    score_pos  = np.dot(u_pos, v_c)
    sig_pos    = sigmoid(score_pos)
    loss       = -np.log(sig_pos + 1e-9)
    grad_vc    = -(1 - sig_pos) * u_pos
    grad_u_pos = -(1 - sig_pos) * v_c
    
    # Negative terms: log σ(-u_neg · v_c)
    grad_u_neg = {}
    for ni in neg_indices:
        u_neg     = W_context[ni]
        score_neg = np.dot(u_neg, v_c)
        sig_neg   = sigmoid(-score_neg)
        loss      += -np.log(sig_neg + 1e-9)
        grad_vc   += (1 - sig_neg) * u_neg
        grad_u_neg[ni] = (1 - sig_neg) * v_c
    
    return loss, grad_vc, grad_u_pos, grad_u_neg


def train_w2v_neg_sampling(training_pairs, V, d=20, lr=0.05, epochs=300, K=5):
    """Train Word2Vec Skip-Gram with Negative Sampling."""
    W_center  = np.random.randn(V, d) * 0.01
    W_context = np.random.randn(V, d) * 0.01
    
    ns_vocab, ns_probs = build_neg_sampler(word2idx, CORPUS)
    loss_history = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        shuffled   = training_pairs.copy()
        random.shuffle(shuffled)
        
        for center_idx, context_idx in shuffled:
            neg_idxs = sample_negatives(center_idx, context_idx, ns_vocab, ns_probs, K=K)
            loss, grad_vc, grad_u_pos, grad_u_neg = neg_sampling_loss_and_grads(
                center_idx, context_idx, neg_idxs, W_center, W_context
            )
            total_loss += loss
            
            # Update center vector
            W_center[center_idx] -= lr * grad_vc
            # Update positive context vector
            W_context[context_idx] -= lr * grad_u_pos
            # Update negative context vectors
            for ni, g in grad_u_neg.items():
                W_context[ni] -= lr * g
        
        avg_loss = total_loss / len(shuffled)
        loss_history.append(avg_loss)
        if epoch % 60 == 0 or epoch == epochs-1:
            print(f"  Epoch {epoch+1:4d}/{epochs}  Loss: {avg_loss:.4f}")
    
    return W_center, W_context, loss_history


print("Training Word2Vec Skip-Gram with NEGATIVE SAMPLING (K=5)...")
W_c_ns, W_ctx_ns, loss_hist_ns = train_w2v_neg_sampling(
    training_pairs, V, d=20, lr=0.05, epochs=300, K=5
)
E_w2v_ns = normalize(W_c_ns, norm='l2')

# Compare both training approaches
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_hist,    color=BLUE,   linewidth=2, label='Full Softmax')
ax.plot(loss_hist_ns, color=ORANGE, linewidth=2, label=f'Neg. Sampling (K=5)')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Average Loss', fontsize=11)
ax.set_title('Full Softmax vs Negative Sampling\n(Different loss scales — not directly comparable, but both converge)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/neg_samp_compare.png', dpi=120, bbox_inches='tight')
plt.show()

# Show negative sampling distribution
ns_vocab_list, ns_probs_arr = build_neg_sampler(word2idx, CORPUS)
raw_freq  = np.array([Counter(tokenise(CORPUS)).get(w, 0) for w in ns_vocab_list])
norm_freq = raw_freq / raw_freq.sum()

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(ns_vocab_list))
ax.bar(x - 0.2, norm_freq,   width=0.4, color=BLUE,   alpha=0.8, label='Raw frequency P(w)')
ax.bar(x + 0.2, ns_probs_arr, width=0.4, color=ORANGE, alpha=0.8, label='Smoothed P(w)^0.75')
ax.set_xticks(x); ax.set_xticklabels(ns_vocab_list, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('Negative Sampling Distribution: P(w) vs P(w)^0.75\n(0.75 power boosts rare words, prevents over-sampling frequent ones)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/neg_samp_dist.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6️⃣ Skip-Gram vs CBOW

| | Skip-Gram | CBOW |
|--|-----------|------|
| **Predicts** | Context words from center | Center word from context |
| **Input** | One center word | Sum of context word vectors |
| **Better for** | Rare words | Frequent words |
| **Speed** | Slower | Faster |

> **Analogy:** Skip-Gram is like a guessing game where you give someone a word and they have to name all its neighbours. CBOW is the reverse: you show the neighbours and ask for the word in the middle — like a crossword clue.

In [ ]:
# ── CBOW Implementation ──────────────────────────────────────────
def generate_cbow_pairs(corpus, word2idx, window=2):
    """
    For CBOW: each training example is (context_word_indices, center_word_idx).
    The input is the AVERAGE of context word vectors.
    """
    data = []
    for sentence in corpus:
        words   = sentence.lower().split()
        idx_seq = [word2idx[w] for w in words if w in word2idx]
        for i, center_idx in enumerate(idx_seq):
            context_idxs = []
            for j in range(max(0, i - window), min(len(idx_seq), i + window + 1)):
                if i != j:
                    context_idxs.append(idx_seq[j])
            if context_idxs:
                data.append((context_idxs, center_idx))
    return data

def train_cbow(cbow_data, V, d=20, lr=0.05, epochs=300):
    """Train CBOW model. Input = mean of context vectors."""
    W_ctx    = np.random.randn(V, d) * 0.01   # context → center
    W_center = np.random.randn(V, d) * 0.01   # center word vectors (output layer)
    loss_hist = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        shuffled   = cbow_data.copy()
        random.shuffle(shuffled)
        
        for ctx_idxs, center_idx in shuffled:
            # h = mean of context vectors
            h      = W_ctx[ctx_idxs].mean(axis=0)      # shape (d,)
            scores = W_center @ h                       # shape (V,)
            probs  = softmax(scores)
            loss   = -np.log(probs[center_idx] + 1e-9)
            total_loss += loss
            
            # Gradient w.r.t. output vectors
            delta      = probs.copy()
            delta[center_idx] -= 1.0
            grad_W_center = np.outer(delta, h)
            
            # Gradient w.r.t. h, then distribute to context vectors
            grad_h = W_center.T @ delta               # shape (d,)
            for ci in ctx_idxs:
                W_ctx[ci] -= lr * grad_h / len(ctx_idxs)
            
            W_center -= lr * grad_W_center
        
        avg_loss = total_loss / len(shuffled)
        loss_hist.append(avg_loss)
    
    return W_ctx, W_center, loss_hist

cbow_data = generate_cbow_pairs(CORPUS, word2idx, window=2)
print(f"CBOW training examples: {len(cbow_data)}")
print("Training CBOW...")
W_ctx_cbow, W_center_cbow, loss_hist_cbow = train_cbow(cbow_data, V, d=20, lr=0.05, epochs=300)
E_cbow = normalize(W_ctx_cbow, norm='l2')

# Visual comparison: Skip-Gram vs CBOW similarity for key word pairs
key_pairs = [('dog','cat'), ('king','queen'), ('man','woman'), ('bread','fish'),
             ('dog','king'), ('cat','bread'), ('king','man')]

sg_sims   = []
cbow_sims = []
pair_labels = []
for w1, w2 in key_pairs:
    if w1 in word2idx and w2 in word2idx:
        i1, i2 = word2idx[w1], word2idx[w2]
        sg_sims.append(float(E_w2v[i1] @ E_w2v[i2]))
        cbow_sims.append(float(E_cbow[i1] @ E_cbow[i2]))
        pair_labels.append(f"{w1}↔{w2}")

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(pair_labels))
bars1 = ax.bar(x - 0.2, sg_sims,   width=0.38, color=BLUE,   alpha=0.85, label='Skip-Gram')
bars2 = ax.bar(x + 0.2, cbow_sims, width=0.38, color=ORANGE, alpha=0.85, label='CBOW')
ax.set_xticks(x); ax.set_xticklabels(pair_labels, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Cosine Similarity', fontsize=11)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Skip-Gram vs CBOW: Pairwise Cosine Similarities\n(Related words should score high; unrelated should score low)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/sg_vs_cbow.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7️⃣ GloVe: Global Vectors

GloVe uses **global co-occurrence statistics** to train word vectors. Instead of iterating over individual word-context pairs, it regresses on the log co-occurrence counts of all word pairs.

$$J = \sum_{w,c} f(N(w,c)) \left( v_w^T \tilde{v}_c + b_w + \tilde{b}_c - \log N(w,c) \right)^2$$

Where $f(x)$ is a weighting function that down-weights very frequent pairs and ignores zero-count pairs.

> **Analogy:** GloVe is like fitting a curve to census data. Instead of interviewing one person at a time (Word2Vec), you get the government's pre-aggregated statistics on the entire population (co-occurrence matrix) and fit a model to those aggregate numbers.

In [ ]:
# ── GloVe Implementation ─────────────────────────────────────────
def glove_weight_func(x, x_max=100, alpha=0.75):
    """
    GloVe weighting function f(x):
    - Zero for zero counts (ignore pairs that never co-occur)
    - Grows sublinearly up to x_max (cap frequent pairs)
    - = 1 for counts >= x_max
    """
    return np.where(x < x_max, (x / x_max) ** alpha, 1.0)

def train_glove(M_cooc, V, d=20, lr=0.05, epochs=300):
    """
    Train GloVe vectors from a co-occurrence matrix.
    
    Objective: minimise Σ f(M_ij) (v_i · u_j + b_i + b_j - log M_ij)^2
    """
    # Initialise word + context vectors and biases
    W  = np.random.randn(V, d) * 0.1    # word vectors
    U  = np.random.randn(V, d) * 0.1    # context vectors  
    bw = np.zeros(V)                     # word biases
    bu = np.zeros(V)                     # context biases
    
    # Precompute non-zero entries
    nonzero_i, nonzero_j = np.where(M_cooc > 0)
    log_M = np.log(M_cooc + 1e-9)
    weights = glove_weight_func(M_cooc)
    
    loss_hist = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        # Shuffle non-zero pairs
        order = np.random.permutation(len(nonzero_i))
        
        for k in order:
            i, j = nonzero_i[k], nonzero_j[k]
            fij   = weights[i, j]
            if fij == 0:
                continue
            
            # Residual: v_i · u_j + b_i + b_j - log M_ij
            r     = np.dot(W[i], U[j]) + bw[i] + bu[j] - log_M[i, j]
            loss  = fij * r * r
            total_loss += loss
            
            # Gradients
            g     = 2 * fij * r
            gW_i  = g * U[j]
            gU_j  = g * W[i]
            gbw_i = g
            gbu_j = g
            
            W[i]  -= lr * gW_i
            U[j]  -= lr * gU_j
            bw[i] -= lr * gbw_i
            bu[j] -= lr * gbu_j
        
        avg_loss = total_loss / max(len(order), 1)
        loss_hist.append(avg_loss)
        if epoch % 60 == 0 or epoch == epochs-1:
            print(f"  Epoch {epoch+1:4d}/{epochs}  Loss: {avg_loss:.4f}")
    
    # GloVe averages word and context vectors
    return normalize((W + U) / 2, norm='l2'), loss_hist

print("Training GloVe...")
E_glove, loss_hist_glove = train_glove(M_raw, V, d=20, lr=0.05, epochs=300)

# Visualise the weighting function
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x_vals  = np.linspace(0, 150, 300)
weights = glove_weight_func(x_vals)
axes[0].plot(x_vals, weights, color=ACCENT, linewidth=2.5)
axes[0].axvline(100, color=ORANGE, linestyle='--', alpha=0.7, label='x_max=100')
axes[0].set_xlabel('Co-occurrence count N(w,c)', fontsize=11)
axes[0].set_ylabel('Weight f(N)', fontsize=11)
axes[0].set_title('GloVe Weighting Function f(x)\n(Rare pairs get less weight; very frequent pairs capped at 1)', fontsize=11, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(loss_hist_glove, color=PURPLE, linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Average Loss')
axes[1].set_title('GloVe Training Curve', fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/glove.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8️⃣ Word Analogies: Linear Structure in Embedding Space

One of the most stunning properties of well-trained embeddings: **semantic relationships are (approximately) linear**.

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$

This works because the "gender direction" (man→woman) is consistent throughout the vector space.

> **Analogy:** Imagine a 2D map where "moving north" always means increasing latitude, regardless of starting city. In word vector space, "moving in the royalty direction" consistently means adding the royalty concept to any human-related word.

In [ ]:
# ── Analogies ────────────────────────────────────────────────────
def analogy(a, b, c, embeddings, word2idx, idx2word, topk=5):
    """
    Solve: a is to b as c is to ?
    Answer: d = argmax cos(v_d, v_b - v_a + v_c)
    Excludes a, b, c from candidates.
    """
    words = [a, b, c]
    if any(w not in word2idx for w in words):
        missing = [w for w in words if w not in word2idx]
        print(f"  [Not in vocab: {missing}]")
        return []
    
    va, vb, vc = [embeddings[word2idx[w]] for w in words]
    query      = vb - va + vc
    query      = query / (np.linalg.norm(query) + 1e-9)
    
    sims   = embeddings @ query
    exclude = {word2idx[w] for w in words}
    ranked = sorted([(i, sims[i]) for i in range(len(sims)) if i not in exclude],
                    key=lambda x: -x[1])
    return [(idx2word[i], s) for i, s in ranked[:topk]]

# Test analogies across all 4 embedding methods
embedding_sets = {
    'LSA (PPMI+SVD)': E_lsa,
    'Word2Vec SG'   : E_w2v,
    'W2V Neg.Samp.' : E_w2v_ns,
    'GloVe'         : E_glove,
    'CBOW'          : E_cbow,
}

analogy_tests = [
    ('man', 'king',   'woman',   'woman is to ??? as man is to king  →  expected: queen'),
    ('king', 'man',   'queen',   'queen is to ??? as king is to man  →  expected: woman'),
    ('dog', 'animal', 'cat',     'cat is to ??? as dog is to animal  →  expected: animal'),
]

print("🔍 Analogy Tests: a - b + c = ?")
print("="*70)
for a, b, c, desc in analogy_tests:
    print(f"\n  Query: {a} - {b} + {c}  ({desc})")
    for name, E in embedding_sets.items():
        results = analogy(a, b, c, E, word2idx, idx2word, topk=3)
        res_str = ', '.join(f"{w}({s:.2f})" for w, s in results) if results else '—'
        print(f"    {name:<18}: {res_str}")

# Visualise the analogy geometry for king-queen
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax_i, (name, E) in enumerate(list(embedding_sets.items())[1:3]):
    ax = axes[ax_i]
    # Project to 2D using PCA on the 4 key words
    words4 = ['king', 'queen', 'man', 'woman']
    valid4 = [w for w in words4 if w in word2idx]
    
    if len(valid4) >= 3:
        vecs4 = np.array([E[word2idx[w]] for w in valid4])
        if vecs4.shape[1] > 2:
            pca = PCA(n_components=2); vecs2d = pca.fit_transform(vecs4)
        else:
            vecs2d = vecs4
        
        colors4 = [ORANGE if w in ['king','queen'] else BLUE for w in valid4]
        for j, (w, xy) in enumerate(zip(valid4, vecs2d)):
            ax.scatter(*xy, s=200, color=colors4[j], zorder=5)
            ax.annotate(w, xy, xytext=(8,4), textcoords='offset points', fontsize=12, fontweight='bold')
        
        # Draw parallelogram arrows
        pair_map = {'king': 'queen', 'man': 'woman'}
        word_pos = {w: vecs2d[j] for j, w in enumerate(valid4)}
        for w1, w2 in pair_map.items():
            if w1 in word_pos and w2 in word_pos:
                ax.annotate('', xy=word_pos[w2], xytext=word_pos[w1],
                           arrowprops=dict(arrowstyle='->', color=RED, lw=2))
        if 'king' in word_pos and 'man' in word_pos:
            ax.annotate('', xy=word_pos['king'], xytext=word_pos['man'],
                       arrowprops=dict(arrowstyle='->', color=GREEN, lw=2, linestyle='dashed'))
        if 'queen' in word_pos and 'woman' in word_pos:
            ax.annotate('', xy=word_pos['queen'], xytext=word_pos['woman'],
                       arrowprops=dict(arrowstyle='->', color=GREEN, lw=2, linestyle='dashed'))
    
    ax.set_title(f'{name}\nParallelogram Analogy Geometry', fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(alpha=0.3)

plt.suptitle('"king - man + woman ≈ queen" Visualised in 2D PCA Space', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/analogy_geometry.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9️⃣ t-SNE Visualisation: Seeing the Semantic Space

t-SNE (t-distributed Stochastic Neighbour Embedding) projects high-dimensional vectors into 2D while preserving **local structure** — nearby points stay nearby.

This lets us literally *see* whether our word vectors have learned sensible semantic clusters.

> **Analogy:** t-SNE is like creating a 2D city map from a 20-dimensional dataset of "how similar are these cities?" Cities that are semantically close (Berlin/Munich vs. Tokyo/Osaka) should cluster together even if the raw coordinates don't show it.

In [ ]:
# ── t-SNE Visualisation ──────────────────────────────────────────
from sklearn.manifold import TSNE

def plot_embeddings_2d(embeddings, vocab_words, word2idx, title, ax, categories=None):
    """
    Project embeddings to 2D (t-SNE if dim > 2, else PCA) and plot.
    """
    valid_words = [w for w in vocab_words if w in word2idx]
    vecs = np.array([embeddings[word2idx[w]] for w in valid_words])
    
    if vecs.shape[1] >= 3:
        # First reduce with PCA if > 10 dims (faster t-SNE)
        if vecs.shape[1] > 10:
            pca = PCA(n_components=min(10, vecs.shape[0]-1))
            vecs_pca = pca.fit_transform(vecs)
        else:
            vecs_pca = vecs
        perplexity = min(5, len(valid_words) - 1)
        tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
        coords = tsne.fit_transform(vecs_pca)
    else:
        coords = vecs
    
    for i, word in enumerate(valid_words):
        color = get_color(word)
        ax.scatter(coords[i, 0], coords[i, 1], s=120, color=color, zorder=4, alpha=0.8)
        ax.annotate(word, (coords[i, 0], coords[i, 1]),
                   xytext=(5, 5), textcoords='offset points', fontsize=8.5,
                   color=color, fontweight='bold')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Dimension 1'); ax.set_ylabel('Dimension 2')
    ax.grid(alpha=0.2)
    return coords

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

methods = [
    ('LSA (PPMI + SVD)', E_lsa),
    ('Word2Vec Skip-Gram', E_w2v),
    ('Word2Vec Neg. Sampling', E_w2v_ns),
    ('GloVe', E_glove),
]

for ax, (name, E) in zip(axes, methods):
    plot_embeddings_2d(E, vocab_words, word2idx, f't-SNE: {name}', ax)

# Legend
legend_patches = [mpatches.Patch(color=c, label=cat) for cat, c in CAT_COLORS.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=10, 
           bbox_to_anchor=(0.5, -0.01), title='Semantic Category')

plt.suptitle('Word Embeddings in 2D Space (t-SNE)\nSame-colour words should cluster together if embeddings capture meaning', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/tsne.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n🔍 Look for: Animals (blue) clustering together, royalty (orange) together, food (purple) together.")
print("   If clusters are visible, our embeddings have learned semantic categories from context alone!")

---
## 1️⃣0️⃣ Cross-Lingual Embeddings

If you train embeddings separately in two languages, the internal geometry of the two spaces is remarkably similar. A **linear transformation** can map one to the other.

$$W^* = \arg\min_W \sum_i \|W z_i - x_i\|^2$$

This has a closed-form solution: **Orthogonal Procrustes** (find the best rotation matrix).

> **Analogy:** Imagine two people independently drew maps of the same city — one in English, one in French. Both maps capture the same geography (the same "structure"), just with different labels and rotated. A simple rotation + translation aligns them perfectly.

In [ ]:
# ── Simulated Cross-Lingual Mapping ──────────────────────────────
# We simulate "French" embeddings by applying a rotation + small noise to our English embeddings.
# This mimics the real scenario: two monolingual embedding spaces with similar structure.

np.random.seed(0)

# English embeddings = our GloVe vectors
E_en = E_glove.copy()

# Create a random rotation matrix (simulates a different language)
d    = E_en.shape[1]
Q_true, _ = np.linalg.qr(np.random.randn(d, d))

# "French" = English rotated + small noise (simulate cross-lingual structure)
E_fr = (E_en @ Q_true.T) + np.random.randn(*E_en.shape) * 0.05
E_fr = normalize(E_fr, norm='l2')

# Seed dictionary: a few known word pairs (anchor points)
# In real cross-lingual work, these come from a bilingual dictionary.
SEED_DICT = {
    'dog': 'chien', 'cat': 'chat', 'king': 'roi', 'queen': 'reine',
    'man': 'homme', 'woman': 'femme', 'bread': 'pain', 'fish': 'poisson'
}
# In our simulation, French words map to the SAME indices (since we rotated the same space)
# We use only the seed dictionary indices to learn the mapping
seed_idxs = [word2idx[w] for w in SEED_DICT.keys() if w in word2idx]

def learn_mapping_procrustes(X, Z, seed_idxs):
    """
    Solve Orthogonal Procrustes: find W = argmin ||ZW - X||_F
    subject to W^T W = I  (W is an orthogonal / rotation matrix)
    
    Closed-form solution via SVD of Z^T X:
    Z^T X = U Σ V^T  =>  W = V U^T
    """
    X_seed = X[seed_idxs]   # English vectors for seed words
    Z_seed = Z[seed_idxs]   # French vectors for seed words
    
    M   = Z_seed.T @ X_seed  # d × d matrix
    U, _, VT = np.linalg.svd(M)
    W   = U @ VT              # optimal rotation
    return W

# Learn the mapping using only the seed words
W_map = learn_mapping_procrustes(E_en, E_fr, seed_idxs)

# Apply the learned mapping to ALL French vectors
E_fr_aligned = normalize(E_fr @ W_map, norm='l2')

# Now test: for each English word, find its nearest French "translation"
# (In our simulation, correct translation = same index)
sim_cross = E_fr_aligned @ E_en.T   # (V_fr, V_en)

print("Cross-Lingual Retrieval Results (Top-3 English neighbours for each French word):")
print("(Correct translation = word with same index, shown in [brackets])")
print()
for w in ['dog', 'cat', 'king', 'queen', 'bread', 'man']:
    if w in word2idx:
        fr_idx = word2idx[w]   # In our simulation, same index
        sims_row = sim_cross[fr_idx]
        top3 = sorted(enumerate(sims_row), key=lambda x: -x[1])[:3]
        top3_str = ', '.join(f"{idx2word[i]}({s:.3f}){'  ✅' if i == fr_idx else ''}" for i, s in top3)
        print(f"  fr[{SEED_DICT.get(w, w+'_fr'):<8}]  →  {top3_str}")

# Visualise before and after alignment
test_words = ['dog', 'cat', 'king', 'queen', 'man', 'woman']
test_idxs  = [word2idx[w] for w in test_words if w in word2idx]
valid_tw   = [w for w in test_words if w in word2idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (E_target, title) in zip(axes, [
    (E_fr[test_idxs, :2], 'BEFORE Alignment (French vs English)'),
    (E_fr_aligned[test_idxs, :2], 'AFTER Orthogonal Procrustes Alignment'),
]):
    E_en_2d = E_en[test_idxs, :2]
    ax.scatter(E_en_2d[:, 0], E_en_2d[:, 1], color=BLUE, s=180, label='English', zorder=5)
    ax.scatter(E_target[:, 0], E_target[:, 1], color=ORANGE, marker='*', s=250, label='French', zorder=5)
    for i, w in enumerate(valid_tw):
        ax.annotate(w,   (E_en_2d[i,0], E_en_2d[i,1]),   xytext=(4,4), textcoords='offset points', fontsize=9, color=BLUE)
        ax.annotate(SEED_DICT.get(w,w), (E_target[i,0], E_target[i,1]), xytext=(4,-12), textcoords='offset points', fontsize=9, color=ORANGE)
        ax.plot([E_en_2d[i,0], E_target[i,0]], [E_en_2d[i,1], E_target[i,1]], 'gray', alpha=0.3, lw=1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.2)

plt.suptitle('Cross-Lingual Alignment via Orthogonal Procrustes\n(Lines connect corresponding word pairs; after alignment, pairs should be close)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/cross_lingual.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 1️⃣1️⃣ Bias in Word Embeddings

Word embeddings trained on human-generated text **absorb and amplify human biases**. The famous example: `man - doctor + woman ≈ nurse` (Bolukbasi et al., NeurIPS 2016).

We'll demonstrate how to:
1. **Detect** gender bias (project words onto the gender axis)
2. **Debias** embeddings by removing the bias direction

> **Warning:** These biases aren't just academic — they can cause real harm when used in resume screening, loan approval, or medical diagnosis systems.

In [ ]:
# ── Bias Detection & Debiasing ───────────────────────────────────
# Step 1: Find the 'gender direction' in embedding space
# The gender direction = direction from 'man' to 'woman' (or 'king' to 'queen')

E_test = E_w2v  # use our trained Word2Vec embeddings

def get_gender_direction(embeddings, word2idx):
    """Estimate gender axis as the direction (woman - man) normalised."""
    if 'man' not in word2idx or 'woman' not in word2idx:
        return None
    v_man   = embeddings[word2idx['man']]
    v_woman = embeddings[word2idx['woman']]
    direction = v_woman - v_man
    return direction / (np.linalg.norm(direction) + 1e-9)

gender_dir = get_gender_direction(E_test, word2idx)

# Step 2: Project all words onto the gender direction
# Words with high positive projection → female-coded
# Words with high negative projection → male-coded
if gender_dir is not None:
    projections = {w: float(E_test[word2idx[w]] @ gender_dir)
                   for w in vocab_words if w in word2idx}
    sorted_proj = sorted(projections.items(), key=lambda x: x[1])
    
    print("Gender Axis Projections (negative = male-coded, positive = female-coded):")
    print(f"\n  Most 'male-coded':   " + str(sorted_proj[:5]))
    print(f"  Most 'female-coded': " + str(sorted_proj[-5:]))
    
    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    words_to_plot = sorted(projections.keys(), key=lambda w: projections[w])
    proj_vals     = [projections[w] for w in words_to_plot]
    bar_colors    = [BLUE if p < 0 else ORANGE for p in proj_vals]
    
    axes[0].barh(words_to_plot, proj_vals, color=bar_colors, alpha=0.85)
    axes[0].axvline(0, color='black', linewidth=1)
    axes[0].set_xlabel('Projection on Gender Axis (woman − man direction)', fontsize=10)
    axes[0].set_title('Gender Bias in Word Embeddings\n(Blue = male-coded, Orange = female-coded)', fontsize=11, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Step 3: Debiasing — project out the gender direction from 'gender-neutral' words
    # For gender-neutral words (non-gendered words), remove their component along the gender axis
    gender_words = {'man', 'woman', 'king', 'queen', 'prince', 'princess', 'royals', 'humans'}
    neutral_words = [w for w in vocab_words if w not in gender_words]
    
    E_debiased = E_test.copy()
    for w in neutral_words:
        if w in word2idx:
            i = word2idx[w]
            # Remove projection onto gender direction
            E_debiased[i] = E_test[i] - np.dot(E_test[i], gender_dir) * gender_dir
    E_debiased = normalize(E_debiased, norm='l2')
    
    # Show change in projections after debiasing
    proj_after = {w: float(E_debiased[word2idx[w]] @ gender_dir)
                  for w in neutral_words if w in word2idx}
    
    sample_neutrals = [w for w in neutral_words[:12] if w in word2idx]
    x = np.arange(len(sample_neutrals))
    before_vals = [projections[w] for w in sample_neutrals]
    after_vals  = [proj_after[w] for w in sample_neutrals]
    
    axes[1].bar(x - 0.2, before_vals, width=0.38, color=RED,   alpha=0.8, label='Before debiasing')
    axes[1].bar(x + 0.2, after_vals,  width=0.38, color=GREEN, alpha=0.8, label='After debiasing')
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels(sample_neutrals, rotation=40, ha='right', fontsize=9)
    axes[1].set_ylabel('Gender Projection')
    axes[1].set_title('Debiasing Effect on Neutral Words\n(After debiasing, projections → ~0)', fontsize=11, fontweight='bold')
    axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/tmp/bias.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    avg_before = np.mean(np.abs(before_vals))
    avg_after  = np.mean(np.abs(after_vals))
    print(f"\n✅ Average |gender projection| before debiasing: {avg_before:.4f}")
    print(f"   Average |gender projection| after  debiasing: {avg_after:.6f}")
    print(f"   Bias reduced by: {(1 - avg_after/avg_before)*100:.1f}%")

---
## 1️⃣2️⃣ Semantic Change Detection

If the same word appears in two different corpora (different time periods, communities, etc.), we can train embeddings on each corpus and compare them to detect **meaning shifts**.

**Method 1 (ACL 2016):** Align the two embedding spaces, then measure which words' vectors don't match.

**Method 2 (ACL 2020):** Compare nearest-neighbour sets. If a word's neighbours change between corpora, its meaning shifted.

> **Analogy:** If you look up "gay" in a 1950s dictionary vs. a 2020s dictionary, the nearest synonyms are completely different. Nearest-neighbour comparison does exactly this, automatically, at scale.

In [ ]:
# ── Semantic Change Detection ─────────────────────────────────────
# Simulate two corpora: Corpus A (original) vs Corpus B (modified — some words used differently)

CORPUS_A = CORPUS.copy()  # Original corpus

# Corpus B: we modify the context of 'king' and 'market' to simulate meaning shift
# 'king' now appears in food contexts (instead of royalty) — simulating semantic shift
CORPUS_B = CORPUS.copy()
CORPUS_B += [
    "the king of bread is sold at market",
    "king fish is the best fish",
    "market king is a brand of bread",
    "the king bread recipe uses fish and meat",
    "market sells the king brand food",
]

# Remove royalty sentences from corpus B (simulating shift away from royalty meaning)
CORPUS_B = [s for s in CORPUS_B if 'kingdom' not in s and 'royals' not in s]

def train_embeddings_for_corpus(corpus, word2idx, d=15, epochs=250):
    """Train Word2Vec Skip-Gram on a given corpus."""
    pairs = generate_training_pairs(corpus, word2idx, window=2)
    if not pairs:
        return normalize(np.random.randn(len(word2idx), d), norm='l2')
    W_c, W_ctx, _ = train_w2v_skipgram(pairs, len(word2idx), d=d, lr=0.05, epochs=epochs, verbose=False)
    return normalize(W_c, norm='l2')

print("Training embeddings on Corpus A (original)...")
E_A = train_embeddings_for_corpus(CORPUS_A, word2idx, d=15, epochs=250)
print("Training embeddings on Corpus B (modified — 'king' used in food context)...")
E_B = train_embeddings_for_corpus(CORPUS_B, word2idx, d=15, epochs=250)

# Method 1: Align B to A (Orthogonal Procrustes), measure displacement
def align_embeddings(E_source, E_target, align_idxs=None):
    """Find rotation W that minimises ||E_source @ W - E_target||."""
    if align_idxs is not None:
        M = E_source[align_idxs].T @ E_target[align_idxs]
    else:
        M = E_source.T @ E_target
    U, _, VT = np.linalg.svd(M)
    W = U @ VT
    return W

# Align using all words as anchors (in practice, use stable words)
W_align = align_embeddings(E_A, E_B)
E_B_aligned = normalize(E_B @ W_align.T, norm='l2')

# Semantic displacement score (1 - cosine_similarity after alignment)
displacement = {}
for w in vocab_words:
    if w in word2idx:
        i = word2idx[w]
        cos_sim = float(E_A[i] @ E_B_aligned[i])
        displacement[w] = 1 - cos_sim

# Method 2: Neighbour intersection
def neighbour_intersection_score(E1, E2, word_idx, k=5, V=None):
    """Lower intersection = more meaning change."""
    V = V or E1.shape[0]
    sims1  = E1 @ E1[word_idx]
    sims2  = E2 @ E2[word_idx]
    nn1    = set(np.argsort(sims1)[::-1][1:k+1])
    nn2    = set(np.argsort(sims2)[::-1][1:k+1])
    return len(nn1 & nn2) / k   # 0 = completely different neighbours

nn_change = {
    w: 1 - neighbour_intersection_score(E_A, E_B, word2idx[w], k=min(5, V-1))
    for w in vocab_words if w in word2idx
}

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (scores, title, cmap) in zip(axes, [
    (displacement, 'Alignment-Based Semantic Displacement\n(higher = more meaning change)', 'Reds'),
    (nn_change,    'Neighbour Intersection Change Score\n(higher = fewer shared neighbours)', 'Oranges'),
]):
    sorted_scores = sorted(scores.items(), key=lambda x: -x[1])
    words_s = [w for w, _ in sorted_scores]
    vals_s  = [v for _, v in sorted_scores]
    colors_s= [RED if v > np.percentile(vals_s, 75) else GRAY for v in vals_s]
    
    ax.barh(words_s[::-1], vals_s[::-1], color=colors_s[::-1], alpha=0.8)
    ax.axvline(np.mean(vals_s), color='blue', linestyle='--', alpha=0.5, label='Mean')
    ax.set_xlabel('Change Score', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(); ax.grid(axis='x', alpha=0.3)

plt.suptitle('Semantic Change Detection: Corpus A vs Corpus B\n("king" and related words should score high — they changed context)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/semantic_change.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nTop-5 most changed words by alignment method:")
for w, s in sorted(displacement.items(), key=lambda x: -x[1])[:5]:
    print(f"  {w:<10}: {s:.4f}")
print("\nTop-5 most changed words by neighbour method:")
for w, s in sorted(nn_change.items(), key=lambda x: -x[1])[:5]:
    print(f"  {w:<10}: {s:.4f}")

---
## 📊 Final Comparison: All Methods Side by Side

In [ ]:
# ── Comprehensive Method Comparison ──────────────────────────────
def evaluate_embeddings(E, word2idx, idx2word, pairs_same, pairs_diff):
    """
    Simple intrinsic evaluation:
    - avg similarity for 'should be similar' pairs
    - avg similarity for 'should be different' pairs
    - higher gap = better embeddings
    """
    def sim(w1, w2):
        if w1 in word2idx and w2 in word2idx:
            return float(E[word2idx[w1]] @ E[word2idx[w2]])
        return 0.0
    
    avg_same = np.mean([sim(a,b) for a,b in pairs_same if a in word2idx and b in word2idx])
    avg_diff = np.mean([sim(a,b) for a,b in pairs_diff if a in word2idx and b in word2idx])
    return avg_same, avg_diff, avg_same - avg_diff

# Word pairs that SHOULD be similar
similar_pairs = [('dog','cat'), ('king','queen'), ('man','woman'), ('bread','fish'), ('prince','princess')]
# Word pairs that SHOULD be different
different_pairs = [('dog','king'), ('cat','bread'), ('man','fish'), ('queen','dog'), ('bread','king')]

print("\n" + "="*72)
print(f"{'Method':<22} {'Sim(related)':<15} {'Sim(unrelated)':<16} {'Gap (higher=better)':<20}")
print("="*72)
for name, E in embedding_sets.items():
    avg_s, avg_d, gap = evaluate_embeddings(E, word2idx, idx2word, similar_pairs, different_pairs)
    print(f"{name:<22} {avg_s:+.4f}         {avg_d:+.4f}          {gap:+.4f}")
print("="*72)

# Final comprehensive bar chart
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

methods_list = list(embedding_sets.keys())
sim_vals  = [evaluate_embeddings(E, word2idx, idx2word, similar_pairs, different_pairs)[0] for E in embedding_sets.values()]
diff_vals = [evaluate_embeddings(E, word2idx, idx2word, similar_pairs, different_pairs)[1] for E in embedding_sets.values()]
gap_vals  = [evaluate_embeddings(E, word2idx, idx2word, similar_pairs, different_pairs)[2] for E in embedding_sets.values()]

x = np.arange(len(methods_list))
axes[0].bar(x, sim_vals,  color=[GREEN  if v > 0 else RED   for v in sim_vals],  alpha=0.8)
axes[0].set_title('Avg. Similarity\nfor Related Pairs', fontweight='bold')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(methods_list, rotation=30, ha='right', fontsize=9)

axes[1].bar(x, diff_vals, color=[RED   if v > 0.1 else GREEN for v in diff_vals], alpha=0.8)
axes[1].set_title('Avg. Similarity\nfor Unrelated Pairs (lower = better)', fontweight='bold')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(methods_list, rotation=30, ha='right', fontsize=9)

axes[2].bar(x, gap_vals,  color=[BLUE  if v > 0 else RED   for v in gap_vals],  alpha=0.8)
axes[2].set_title('Gap = Related − Unrelated\n(HIGHER = BETTER)', fontweight='bold')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels(methods_list, rotation=30, ha='right', fontsize=9)

for ax in axes:
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Intrinsic Evaluation: How Well Do Embeddings Capture Semantic Similarity?\n(Gap = ability to distinguish related from unrelated word pairs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/final_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🏁 Summary & Key Takeaways

| Method | Core Idea | When to Use |
|--------|-----------|-------------|
| **One-Hot** | Unique index per word | Never for NLP |
| **Co-occurrence counts** | Count word co-appearances | Baseline comparison |
| **PPMI** | Mutual information of co-occurrence | Count-based benchmark |
| **LSA/SVD** | Reduce PPMI matrix dimensionally | When you also need doc vectors |
| **Word2Vec SG** | Predict context from center | General-purpose, most popular |
| **Word2Vec NS** | Efficient SG with sampled negatives | Large corpora, production |
| **CBOW** | Predict center from context | Faster training, large vocab |
| **GloVe** | Regress on global co-occurrence counts | When global statistics matter |

### 🔑 The Three Deep Insights

1. **Distributional hypothesis** — meaning comes from context. All methods here operationalize this.
2. **Linear geometry** — semantic relationships are directions in vector space. This is why analogies work.
3. **Embeddings inherit biases** — trained on human text, embeddings reflect human prejudices. Always audit.

---

*Continue learning: [Lena Voita's NLP Course](https://lena-voita.github.io/nlp_course.html)*